# Credit Risk Modeling: Loan Default Prediction
**Project type:** Data analysis + supervised machine learning  
**Dataset:** Synthetic demonstration data generated for this repository  
**Target:** `default` (1 = simulated default, 0 = simulated non-default)

> **Important:** This is a learning project using synthetic data. Results do not represent a real lender, actual borrowers, or a production-ready credit decision system.

## 1. Business problem
Lenders need to estimate repayment risk while balancing missed defaults against incorrectly flagging applicants who would repay. This notebook demonstrates a reproducible workflow: data quality checks, exploratory analysis, train/test split, preprocessing, model comparison, evaluation, and feature interpretation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

RANDOM_STATE = 42
df = pd.read_csv("data/credit_risk_synthetic.csv")
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data quality checks

In [ ]:
print("Data types and non-null counts:")
display(df.info())
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nTarget distribution:")
display(df["default"].value_counts().rename(index={0:"Non-default",1:"Default"}).to_frame("records"))
print("\\nSummary statistics:")
display(df.describe(include="all").T)

## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="default", ax=axes[0])
axes[0].set_title("Simulated loan outcome counts")
axes[0].set_xticks([0,1])\naxes[0].set_xticklabels(["Non-default (0)", "Default (1)"])
sns.histplot(data=df, x="interest_rate", hue="default", bins=25, element="step",
             stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Interest rate distribution by outcome")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="default", y="debt_to_income_ratio")
plt.title("Debt-to-income ratio by simulated outcome")
plt.xticks([0,1], ["Non-default", "Default"])
plt.tight_layout()
plt.show()

numeric = df.select_dtypes(include=np.number)
plt.figure(figsize=(9,6))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric feature correlation matrix")
plt.tight_layout()
plt.show()

## 4. Prepare data without leakage
The target is separated from predictors before splitting. Imputation, scaling, and one-hot encoding are fitted only on the training data through a scikit-learn pipeline. Outcome-defining information is not used as a feature.

In [ ]:
target = "default"
X = df.drop(columns=[target])
y = df[target]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training default rate:", round(y_train.mean(), 3))
print("Test default rate:", round(y_test.mean(), 3))

## 5. Train and compare models
Logistic Regression provides an interpretable baseline. Decision Tree and Random Forest provide non-linear comparisons. Class weighting is used to account for imbalance without synthesizing examples. If SMOTE is required by the project brief, it should be applied only inside the training workflow and never to the test set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=3,
                                            class_weight="balanced", random_state=RANDOM_STATE,
                                            n_jobs=-1)
}
fitted = {}
rows = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    scores = pipe.predict_proba(X_test)[:, 1]
    fitted[name] = pipe
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, scores)
    })
results = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
display(results.round(3))

## 6. Evaluate the selected model

In [ ]:
best_name = results.iloc[0]["model"]
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)
best_scores = best_model.predict_proba(X_test)[:, 1]
print("Selected by test ROC-AUC for demonstration:", best_name)
print(classification_report(y_test, best_pred, target_names=["Non-default", "Default"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, display_labels=["Non-default","Default"], cmap="Blues")
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

## 7. Feature importance (permutation importance)
Permutation importance measures how much a metric drops when one input column is shuffled. It provides a model-specific predictive signal, not proof of causation. The following computation can take a short time because it evaluates the full pipeline repeatedly.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5,
                              random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance)
sns.barplot(data=importance.head(10), x="importance_mean", y="feature")
plt.title("Top predictive features — permutation importance")
plt.xlabel("Mean decrease in ROC-AUC when shuffled")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 8. Conclusions and limitations
Complete this section after running the notebook. Discuss the metrics and confusion matrix, the trade-off between precision and recall, and which variables appear useful in this synthetic dataset.

**Limitations:** Data is simulated; the model is not validated for real borrowers. Real lending systems require representative data, fairness and compliance review, probability calibration, monitoring, and human oversight. Do not use this model to make real credit decisions.